# 02 – Cleaning and Feature Engineering (Full Dataset)

Builds on `01_define_conflict_zone.ipynb`. Applies a minimal cleaning pipeline (no anomaly-removing filters — keep anomalies!) and computes all features on the full ~2M-row dataset.

## Cleaning

1. **Type-aware**: vehicles (Type ∈ {2, 3}) are the primary analysis subjects; VRUs (Type ∈ {0, 1}) are kept for the `dist_nearest_m` feature.
2. **Minimum trajectory length**: vehicles with fewer than 10 points cannot have features computed reliably.
3. **Sort and deduplicate**: ensure (ID, Time) is unique and ordered.
4. **Bounding-box sanity**: drop any out-of-range coordinates (rare).

The conflict-zone polygon enters as a *feature* (`in_conflict_zone`), not a filter.

## Feature set 

1. Speed (recorded, km/h and m/s)
2. Displacement-implied speed (diagnostic)
3. Acceleration
4. Heading, heading change, yaw rate
5. Distance to nearest other road user
6. Deviation from cluster-median path
7. Time in conflict zone (running and total)
8. Window statistics (1 s, 2 s rolling mean/std/absmax of speed, acceleration, heading change)

In [1]:
import numpy as np
import pandas as pd
import time
from pathlib import Path
from matplotlib.path import Path as MplPath
from scipy.spatial import cKDTree

from conflict_zone_polygon import CONFLICT_ZONE_POLYGON

## 1. Parameters

In [2]:
CSV_PATH = Path('Slottskogsgatan_tracks_extramerged.csv') 
OUT_PATH = Path('features_full.parquet')

VEHICLE_TYPES = {2, 3}        # primary subjects of anomaly detection
MIN_POINTS_PER_ID = 10        # vehicles with fewer points are dropped (matches WorkProcess)

MIN_DT_S = 0.1                # below this, derivative features get NaN (noise amplification)
MAX_DT_S = 0.5                # above this, derivative features get NaN (gap too large)
MIN_DISP_M = 0.05             # below this displacement, heading is undefined

# Distance-to-nearest: which other-road-user types count as neighbours?
NEIGHBOUR_TYPES = {0, 1, 2, 3}   # all types — VRUs matter for safety
BUCKET_S = 0.25                  # time-bucket width for snapping timestamps

# Deviation-from-path: entry/exit sector clustering
CENTRE_XY = (5.0, -15.0)      # approximate centre of the conflict zone
N_SECTORS = 8                 # 45° sectors
N_REF_POINTS = 50             # resample each trajectory to this many arc-length points
MIN_TRAJ_PER_CLUSTER = 50     # corridors with fewer trajectories don't get a defined median

WINDOWS = ['1s', '2s']
WIN_BASE_COLS = ['Speed_ms', 'acceleration_ms2', 'heading_change_rad']

## 2. Load and clean

In [3]:
t0 = time.time()
df = pd.read_csv(CSV_PATH, sep=';')
print(f'Loaded {len(df):,} rows in {time.time()-t0:.1f}s')

df['Time'] = pd.to_datetime(df['Time'], format='ISO8601')
df = df.sort_values(['ID', 'Time']).reset_index(drop=True)

# Drop exact (ID, Time) duplicates if any.
before = len(df)
df = df.drop_duplicates(subset=['ID', 'Time'], keep='first')
print(f'Dropped {before - len(df):,} exact (ID, Time) duplicates')

# Drop NaN coordinates (shouldn't happen but to be safe).
before = len(df)
df = df.dropna(subset=['X', 'Y', 'Speed'])
print(f'Dropped {before - len(df):,} rows with NaN X/Y/Speed')

# Type-aware separation: vehicles for main analysis, VRUs kept as neighbours.
vrus = df[df['Type'].isin({0, 1})].copy()
veh = df[df['Type'].isin(VEHICLE_TYPES)].copy()
print(f'\nVehicles: {len(veh):,} rows, {veh["ID"].nunique():,} IDs')
print(f'VRUs:     {len(vrus):,} rows, {vrus["ID"].nunique():,} IDs (kept for neighbour distance)')

# Minimum-length filter on vehicles.
id_counts = veh.groupby('ID').size()
keep_ids = id_counts[id_counts >= MIN_POINTS_PER_ID].index
before = len(veh)
veh = veh[veh['ID'].isin(keep_ids)].reset_index(drop=True)
print(f'\nKept {veh["ID"].nunique():,} vehicles with ≥{MIN_POINTS_PER_ID} points ({len(veh):,} rows, {before-len(veh):,} dropped)')

Loaded 1,978,476 rows in 1.6s
Dropped 0 exact (ID, Time) duplicates
Dropped 0 rows with NaN X/Y/Speed

Vehicles: 1,437,096 rows, 56,133 IDs
VRUs:     541,380 rows, 14,980 IDs (kept for neighbour distance)

Kept 46,578 vehicles with ≥10 points (1,372,369 rows, 64,727 dropped)


## 3. Speed variants and base kinematics

`Speed_kmh` (original), `Speed_ms` (m/s, used for derivatives), and `Speed_ms_displ` (displacement-implied, diagnostic only).

In [4]:
veh['Speed_kmh'] = veh['Speed']
veh['Speed_ms'] = veh['Speed'] / 3.6

g = veh.groupby('ID', sort=False)
veh['dt_s'] = g['Time'].diff().dt.total_seconds()
veh['dx'] = g['X'].diff()
veh['dy'] = g['Y'].diff()
veh['displacement_m'] = np.hypot(veh['dx'], veh['dy'])

valid_dt = veh['dt_s'].between(MIN_DT_S, MAX_DT_S, inclusive='both')
veh['Speed_ms_displ'] = np.where(valid_dt, veh['displacement_m'] / veh['dt_s'], np.nan)

print(f'Valid-dt rows: {valid_dt.sum():,} / {len(veh):,} = {valid_dt.mean()*100:.1f}%')

Valid-dt rows: 1,278,163 / 1,372,369 = 93.1%


## 4. Acceleration

In [5]:
veh['d_speed_ms'] = g['Speed_ms'].diff()
veh['acceleration_ms2'] = np.where(valid_dt, veh['d_speed_ms'] / veh['dt_s'], np.nan)

veh['prev_estimated'] = g['Estimated'].shift(1)
veh['accel_from_estimated'] = (veh['Estimated'] == 1) | (veh['prev_estimated'] == 1)

print('Acceleration (m/s²):')
print(veh['acceleration_ms2'].describe(percentiles=[.001, .01, .5, .99, .999]).round(3))
print(f'\n|a| > 10 m/s² (suspicious): {(veh["acceleration_ms2"].abs() > 10).sum():,}')
print(f'|a| > 20 m/s² (impossible):  {(veh["acceleration_ms2"].abs() > 20).sum():,}')
print(f'  of which from-estimated:   {((veh["acceleration_ms2"].abs() > 20) & veh["accel_from_estimated"]).sum():,}')

Acceleration (m/s²):
count    1278163.000
mean           0.032
std            3.483
min          -92.836
0.1%         -26.680
1%           -13.985
50%            0.114
99%            9.624
99.9%         23.556
max          111.011
Name: acceleration_ms2, dtype: float64

|a| > 10 m/s² (suspicious): 33,293
|a| > 20 m/s² (impossible):  6,286
  of which from-estimated:   5,273


## 5. Heading, heading change, yaw rate

In [6]:
raw_heading = np.arctan2(veh['dy'], veh['dx'])
veh['heading_rad'] = np.where(veh['displacement_m'] >= MIN_DISP_M, raw_heading, np.nan)

def _unwrap_group(s):
    mask = s.notna()
    if mask.sum() < 2:
        return s
    out = s.copy()
    out.loc[mask] = np.unwrap(s.loc[mask].to_numpy())
    return out

veh['heading_unwrapped'] = g['heading_rad'].transform(_unwrap_group)
veh['heading_change_rad'] = g['heading_unwrapped'].diff()
veh['yaw_rate_rads'] = np.where(valid_dt, veh['heading_change_rad'] / veh['dt_s'], np.nan)

print(veh[['heading_rad', 'heading_change_rad', 'yaw_rate_rads']].describe().round(3))

       heading_rad  heading_change_rad  yaw_rate_rads
count  1320261.000         1271655.000    1227105.000
mean        -0.059               0.005          0.019
std          1.418               0.213          0.937
min         -3.142              -3.142        -31.025
25%         -1.214              -0.019         -0.071
50%         -0.864               0.005          0.020
75%          1.724               0.041          0.157
max          3.141               3.141         31.386


## 6. Distance to nearest other road user

Pairwise within each time bucket. Vehicles compared against vehicles AND VRUs (since `NEIGHBOUR_TYPES = {0, 1, 2, 3}`). This is O(B × k log k) per bucket where B is bucket count and k is the typical bucket population.

In [7]:
# Build neighbour pool (vehicles + VRUs combined). Bucket timestamps to align disparate clocks.
neighbours = pd.concat([veh[['ID', 'Time', 'X', 'Y', 'Type']],
                        vrus[['ID', 'Time', 'X', 'Y', 'Type']]],
                       ignore_index=True)
neighbours = neighbours[neighbours['Type'].isin(NEIGHBOUR_TYPES)]

t0_ref = neighbours['Time'].min()
neighbours['bucket'] = ((neighbours['Time'] - t0_ref).dt.total_seconds() // BUCKET_S).astype('int64')
veh['bucket'] = ((veh['Time'] - t0_ref).dt.total_seconds() // BUCKET_S).astype('int64')

# Sort by bucket so we can index by contiguous slices.
neighbours = neighbours.sort_values('bucket').reset_index(drop=True)
n_bucket_arr = neighbours['bucket'].to_numpy()
n_x = neighbours['X'].to_numpy()
n_y = neighbours['Y'].to_numpy()
n_id = neighbours['ID'].to_numpy()

# Build bucket → (start, end) index into the sorted neighbours array.
change = np.concatenate([[True], np.diff(n_bucket_arr) != 0])
starts = np.flatnonzero(change)
ends = np.concatenate([starts[1:], [len(neighbours)]])
bucket_to_range = {b: (s, e) for b, s, e in zip(n_bucket_arr[starts], starts, ends)}
print(f'Buckets indexed: {len(bucket_to_range):,}')

# Sort vehicle rows by bucket too, so contiguous slices of the same bucket process together.
v_x = veh['X'].to_numpy()
v_y = veh['Y'].to_numpy()
v_id = veh['ID'].to_numpy()
v_bucket = veh['bucket'].to_numpy()

veh_order = np.argsort(v_bucket, kind='stable')
v_x_s = v_x[veh_order]; v_y_s = v_y[veh_order]
v_id_s = v_id[veh_order]; v_bucket_s = v_bucket[veh_order]

v_change = np.concatenate([[True], np.diff(v_bucket_s) != 0])
v_starts = np.flatnonzero(v_change)
v_ends = np.concatenate([v_starts[1:], [len(veh)]])

nearest_sorted = np.full(len(veh), np.nan)

t0 = time.time()
for i in range(len(v_starts)):
    b = v_bucket_s[v_starts[i]]
    rng = bucket_to_range.get(b)
    if rng is None: continue
    ns, ne = rng
    if ne - ns < 2: continue
    nx = n_x[ns:ne]; ny = n_y[ns:ne]; nid = n_id[ns:ne]
    vs, ve = v_starts[i], v_ends[i]
    vx = v_x_s[vs:ve]; vy = v_y_s[vs:ve]; vid_local = v_id_s[vs:ve]
    # Pairwise distances inside this bucket only.
    d = np.hypot(vx[:, None] - nx[None, :], vy[:, None] - ny[None, :])
    d = np.where(vid_local[:, None] == nid[None, :], np.inf, d)
    row_min = d.min(axis=1)
    nearest_sorted[vs:ve] = np.where(np.isfinite(row_min), row_min, np.nan)

# Unsort to original order and attach.
nearest = np.full(len(veh), np.nan)
nearest[veh_order] = nearest_sorted
veh['dist_nearest_m'] = nearest
print(f'Done in {time.time()-t0:.1f}s.')
print(f'Valid distances: {np.isfinite(nearest).sum():,} / {len(nearest):,} ({np.isfinite(nearest).mean()*100:.1f}%)')
print(f'Distribution (m): median={np.nanmedian(nearest):.2f}, 1%={np.nanpercentile(nearest, 1):.2f}, 99%={np.nanpercentile(nearest, 99):.2f}')

Buckets indexed: 605,513
Done in 3.3s.
Valid distances: 1,261,502 / 1,372,369 (91.9%)
Distribution (m): median=13.29, 1%=0.01, 99%=62.31


## 7. Deviation from cluster-median path

Cluster trajectories by entry/exit sector (around the conflict-zone centre), build a median path per cluster, measure each point's distance to its cluster's median.

In [8]:
cx, cy = CENTRE_XY

def sector_of(x, y, n=N_SECTORS):
    a = np.arctan2(y - cy, x - cx)
    return ((a + 2 * np.pi) % (2 * np.pi) / (2 * np.pi / n)).astype(int)

first_last = veh.groupby('ID', sort=False).agg(
    x_first=('X', 'first'), y_first=('Y', 'first'),
    x_last=('X', 'last'),   y_last=('Y', 'last'),
)
first_last['entry_sector'] = sector_of(first_last['x_first'].to_numpy(), first_last['y_first'].to_numpy())
first_last['exit_sector']  = sector_of(first_last['x_last'].to_numpy(),  first_last['y_last'].to_numpy())
first_last['cluster'] = list(zip(first_last['entry_sector'], first_last['exit_sector']))

def resample(sub, n=N_REF_POINTS):
    xy = sub[['X', 'Y']].to_numpy()
    if len(xy) < 2: return None
    seg = np.linalg.norm(np.diff(xy, axis=0), axis=1)
    arc = np.concatenate([[0], np.cumsum(seg)])
    if arc[-1] == 0: return None
    u = np.linspace(0, arc[-1], n)
    return np.column_stack([np.interp(u, arc, xy[:, 0]), np.interp(u, arc, xy[:, 1])])

print('Resampling trajectories...')
t0 = time.time()
resampled = {vid: resample(sub) for vid, sub in veh.groupby('ID', sort=False)}
print(f'  done in {time.time()-t0:.1f}s')
first_last['resampled_ok'] = [resampled[vid] is not None for vid in first_last.index]

# Cluster medians.
cluster_median = {}
cluster_sizes = first_last['cluster'].value_counts()
for cluster_id, members in first_last[first_last['resampled_ok']].groupby('cluster'):
    if len(members) < MIN_TRAJ_PER_CLUSTER:
        continue
    stack = np.stack([resampled[vid] for vid in members.index])
    cluster_median[cluster_id] = np.median(stack, axis=0)

print(f'\nClusters with ≥{MIN_TRAJ_PER_CLUSTER} trajectories: {len(cluster_median)}')
print(f'Top 10 cluster sizes: {cluster_sizes.head(10).to_dict()}')
covered = sum(cluster_sizes.get(c, 0) for c in cluster_median)
print(f'Trajectories with a defined cluster median: {covered:,} / {len(first_last):,} = {covered/len(first_last)*100:.1f}%')

Resampling trajectories...
  done in 9.7s

Clusters with ≥50 trajectories: 24
Top 10 cluster sizes: {(2, 7): 10223, (5, 2): 9449, (2, 2): 5833, (2, 1): 4096, (2, 6): 4009, (4, 2): 3103, (3, 2): 2050, (5, 7): 1607, (7, 6): 1420, (7, 7): 735}
Trajectories with a defined cluster median: 46,373 / 46,578 = 99.6%


In [9]:
# Point-to-polyline distance via KDTree on the median's vertices (nearest-vertex approx).
from scipy.spatial import cKDTree as _Tree
cluster_tree = {cid: _Tree(med) for cid, med in cluster_median.items()}
vid_to_cluster = first_last['cluster'].to_dict()

deviation = np.full(len(veh), np.nan)
for vid, sub in veh.groupby('ID', sort=False):
    cid = vid_to_cluster.get(vid)
    tree = cluster_tree.get(cid)
    if tree is None: continue
    dists, _ = tree.query(sub[['X', 'Y']].to_numpy(), k=1)
    deviation[sub.index.to_numpy()] = dists
veh['deviation_m'] = deviation

print('Deviation from cluster median (m):')
print(veh['deviation_m'].describe(percentiles=[.5, .9, .95, .99]).round(2))

Deviation from cluster median (m):
count    1368345.00
mean           2.12
std            3.44
min            0.00
50%            0.84
90%            4.97
95%            9.13
99%           17.98
max           32.34
Name: deviation_m, dtype: float64


## 8. Conflict zone membership and time inside

In [10]:
zone_path = MplPath(CONFLICT_ZONE_POLYGON)
veh['in_conflict_zone'] = zone_path.contains_points(veh[['X', 'Y']].to_numpy())

# Running time inside: only count a segment if both endpoints were inside and dt was valid.
prev_in = g['in_conflict_zone'].shift(1).fillna(False)
segment_inside = prev_in & veh['in_conflict_zone']
veh['dt_in_s'] = np.where(segment_inside & valid_dt, veh['dt_s'], 0.0)
veh['time_in_zone_running_s'] = g['dt_in_s'].cumsum()

totals = veh.groupby('ID')['time_in_zone_running_s'].last().rename('time_in_zone_total_s')
veh = veh.merge(totals, on='ID', how='left')

print(f'Vehicles ever in zone: {(veh.groupby("ID")["in_conflict_zone"].any()).sum():,}')
print(f'\nTime in zone (s):')
print(veh[['time_in_zone_running_s', 'time_in_zone_total_s']].describe().round(2))

Vehicles ever in zone: 38,483

Time in zone (s):
       time_in_zone_running_s  time_in_zone_total_s
count              1372369.00            1372369.00
mean                     1.70                  3.77
std                      2.00                  2.53
min                      0.00                  0.00
25%                      0.00                  2.02
50%                      0.96                  3.79
75%                      3.03                  5.30
max                     19.82                 19.82


## 9. Window statistics (1 s and 2 s rolling)

In [11]:
def _time_rolling_stats(times_s, values, window_s):
    """Compute rolling mean, std, and absmax over a time window for a single group.
    times_s: 1D array of timestamps in seconds (must be sorted, ascending).
    values: 1D array of the same length, may contain NaN.
    window_s: float, window width in seconds (looks back from each row).
    Returns (mean, std, absmax) each as 1D float arrays of length len(values)."""
    n = len(values)
    mean = np.full(n, np.nan, dtype=np.float64)
    std  = np.full(n, np.nan, dtype=np.float64)
    amax = np.full(n, np.nan, dtype=np.float64)
    if n == 0: return mean, std, amax
    # Two-pointer sweep: for each i, find the smallest j such that times_s[i] - times_s[j] <= window_s.
    j = 0
    for i in range(n):
        while times_s[i] - times_s[j] > window_s:
            j += 1
        vs = values[j:i+1]
        # Use nan-aware aggregations; if all-NaN, stays NaN.
        if np.any(~np.isnan(vs)):
            mean[i] = np.nanmean(vs)
            if (i - j) >= 1:
                std[i] = np.nanstd(vs, ddof=1) if np.sum(~np.isnan(vs)) > 1 else 0.0
            amax[i] = np.nanmax(np.abs(vs))
    return mean, std, amax

veh = veh.sort_values(['ID', 'Time']).reset_index(drop=True)
veh['_t_sec'] = (veh['Time'] - veh['Time'].min()).dt.total_seconds().to_numpy()

# Pre-extract numpy arrays once.
id_arr = veh['ID'].to_numpy()
t_arr = veh['_t_sec'].to_numpy()

# Build (start, end) ranges per ID using sorted-by-ID layout.
# Since we already sorted by [ID, Time], all rows of one ID are contiguous.
id_change = np.concatenate([[True], np.diff(id_arr) != 0])
id_starts = np.flatnonzero(id_change)
id_ends = np.concatenate([id_starts[1:], [len(veh)]])
print(f'Vehicle groups: {len(id_starts):,}')

for w_label, w_seconds in [('1s', 1.0), ('2s', 2.0)]:
    print(f'Rolling {w_label}...')
    t_start = time.time()
    out_buffers = {}
    for col in WIN_BASE_COLS:
        out_buffers[col] = (np.full(len(veh), np.nan), np.full(len(veh), np.nan), np.full(len(veh), np.nan))
    for col in WIN_BASE_COLS:
        col_arr = veh[col].to_numpy()
        m_buf, s_buf, a_buf = out_buffers[col]
        for k in range(len(id_starts)):
            s, e = id_starts[k], id_ends[k]
            m, sd, am = _time_rolling_stats(t_arr[s:e], col_arr[s:e], w_seconds)
            m_buf[s:e] = m; s_buf[s:e] = sd; a_buf[s:e] = am
        veh[f'{col}_mean_{w_label}'] = out_buffers[col][0]
        veh[f'{col}_std_{w_label}']  = out_buffers[col][1]
        veh[f'{col}_absmax_{w_label}'] = out_buffers[col][2]
    print(f'  done in {time.time()-t_start:.1f}s')

veh = veh.drop(columns=['_t_sec'])
print('\nWindow columns added:')
for c in veh.columns:
    if any(c.endswith(f'_{w}') for w in WINDOWS):
        print(f'  {c}')

Vehicle groups: 46,578
Rolling 1s...
  done in 131.1s
Rolling 2s...
  done in 131.4s

Window columns added:
  Speed_ms_mean_1s
  Speed_ms_std_1s
  Speed_ms_absmax_1s
  acceleration_ms2_mean_1s
  acceleration_ms2_std_1s
  acceleration_ms2_absmax_1s
  heading_change_rad_mean_1s
  heading_change_rad_std_1s
  heading_change_rad_absmax_1s
  Speed_ms_mean_2s
  Speed_ms_std_2s
  Speed_ms_absmax_2s
  acceleration_ms2_mean_2s
  acceleration_ms2_std_2s
  acceleration_ms2_absmax_2s
  heading_change_rad_mean_2s
  heading_change_rad_std_2s
  heading_change_rad_absmax_2s


## 10. Final table and save

In [12]:
feature_cols = [
    'ID', 'Time', 'X', 'Y', 'Type', 'Estimated',
    'Speed_kmh', 'Speed_ms', 'Speed_ms_displ',
    'dt_s', 'displacement_m', 'acceleration_ms2', 'accel_from_estimated',
    'heading_rad', 'heading_change_rad', 'yaw_rate_rads',
    'dist_nearest_m', 'deviation_m',
    'in_conflict_zone', 'time_in_zone_running_s', 'time_in_zone_total_s',
]
feature_cols += [c for c in veh.columns if any(c.endswith(f'_{w}') for w in WINDOWS)]

out = veh[feature_cols].copy()

# Downcast numerics to save space.
for c in out.select_dtypes('float64').columns:
    out[c] = out[c].astype('float32')
for c in out.select_dtypes('int64').columns:
    out[c] = pd.to_numeric(out[c], downcast='integer')

print(f'Final shape: {out.shape}')
print(f'Memory: {out.memory_usage(deep=True).sum()/1e6:.0f} MB')
print(f'\nNaN fraction per column (top 10 most-NaN):')
print(out.isna().mean().sort_values(ascending=False).head(10).round(3))

out.to_parquet(OUT_PATH, index=False)
print(f'\nSaved to {OUT_PATH}')

Final shape: (1372369, 39)
Memory: 203 MB

NaN fraction per column (top 10 most-NaN):
yaw_rate_rads                   0.106
heading_change_rad_std_1s       0.095
heading_change_rad_std_2s       0.091
dist_nearest_m                  0.081
heading_change_rad              0.073
heading_change_rad_mean_1s      0.071
heading_change_rad_absmax_1s    0.071
heading_change_rad_absmax_2s    0.070
heading_change_rad_mean_2s      0.070
Speed_ms_displ                  0.069
dtype: float64

Saved to features_full.parquet
